In [60]:
import torch
import torch.nn as nn
import torchvision
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from torchvision.datasets import ImageFolder
import os
import numpy as np
from PIL import Image
from collections import Counter

In [ ]:
class_names = ['alef', 'b', 'p', 't', 'se', 'jim', 'che', 'h', 'kh', 'd', 'zal', 'r', 'z', 'zh', 'sin', 'shin', 'sad', 'zad', 't-long', 'z-long', 'ayin', 'ghyin', 'f', 'ghe', 'k', 'g', 'le', 'm', 'n', 'v', 'he', 'ye', 'zero', 'one', 'two', 'three', 'four', 'five', 'six', 'seven', 'eight', 'nine', 'anewfive']

for class_name in class_names:
    if class_name in path_to_image:
        

In [ ]:
#Prepare the dataset
class PersianAlpha(Dataset):
    def __init__(self, path_to_folder):
    
        path_to_alef_files = get_files(path_to_folder, '1-alef')
        path_to_b_files = get_files(path_to_folder, '2-b')
        path_to_p_files = get_files(path_to_folder, '3-p')

        self.training_files = path_to_alef_files + path_to_b_files + path_to_p_files

        self.class_names = ['alef', 'b', 'p', 't', 'se', 'jim', 'che', 'h', 'kh', 'd', 'zal', 'r', 'z', 'zh', 'sin', 'shin', 'sad', 'zad', 't-long', 'z-long', 'ayin', 'ghyin', 'f', 'ghe', 'k', 'g', 'le', 'm', 'n', 'v', 'he', 'ye', 'zero', 'one', 'two', 'three', 'four', 'five', 'six', 'seven', 'eight', 'nine', 'anewfive']

        self.alef_label = 'alef'
        self.b_label = 'b'
        self.p_label = 'p'

        

    def __len__(self):
        return len(self.training_files)

    def __getitem__(self, index):
        path_to_image = self.training_files[index]

        if 'alef' in path_to_image:
            label = self.alef_label
        elif 'b' in path_to_image:
            label = self.b_label
        else:
            label = self.p_label

        img = Image.open(path_to_image)
        return img, label


def get_files(path_to_folder, folder_name):
    path_to_files = []
    for folder in os.scandir(path_to_folder):
        if folder.name == folder_name:
            for file in os.scandir(folder.path):
                if file.is_file():
                    path_to_files.append(file.path)
    return path_to_files    
            
path_to_folder = './alpha/'

In [98]:
#preprocessing
class TransformImg(Dataset):
    def __init__(self, data, transform):
        self.data = data
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        img, label = self.data[index]
        return self.transform(img), label


train_transform = transforms.Compose(
            [
                transforms.Resize((224,224)),
                transforms.RandomHorizontalFlip(p=0.5),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                std=[0.229, 0.224, 0.225]) 
            ]
        )

val_transform = transforms.Compose(
            [
                transforms.Resize((224,224)),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                std=[0.229, 0.224, 0.225]) 
            ]
        )


In [99]:
dataset = PersianAlpha(path_to_folder)

#split the dataset
train_samples = int(0.8*len(dataset))
val_samples = len(dataset) - train_samples

train_split, val_split = torch.utils.data.random_split(dataset, lengths=[train_samples, val_samples], generator=torch.Generator().manual_seed(42))

#preprocess the split
train = TransformImg(train_split, transform=train_transform)
val = TransformImg(val_split, transform=val_transform)

#load the dataset
trainig_dataset = DataLoader(train, batch_size=32, shuffle=True)
validation_dataset = DataLoader(val, batch_size=32, shuffle=True)